# CRL (Charles River Laboratories): Daily News Volume vs. Price (Past Year)

This notebook pulls one year of daily closing prices and daily news article counts for a ticker,
lines them up on the same daily date index, shows them in a table, and plots them on a dual-axis line chart.

**Data sources (both free):**
- **Price:** Yahoo Finance via `yfinance`. No key needed.
- **News:** [Finnhub](https://finnhub.io/register) `company-news` endpoint. The free tier includes about 1 year of company news; you need a free API key.
  If no key is set, the notebook falls back to the **GDELT DOC 2.0 API**, which needs no key. GDELT searches news text for a keyword rather than tagging articles by ticker, so its counts are rougher.

**Rate limits / blocked scraping:** the notebook includes a **snapshot** of CRL data fetched on 2026‑09‑30 (daily close plus GDELT article counts, 2025‑09‑30 to 2026‑09‑30).
- `DATA_MODE = "snapshot"` makes no web requests and always runs.
- `DATA_MODE = "auto"` (the default) tries the live APIs. If a request is rate-limited or fails, it switches to the snapshot.
- Any successful live download is cached to CSV, so running the notebook again does not call the API again.

**How to run in Colab:** choose Runtime → Run all. For live Finnhub news, set `FINNHUB_API_KEY` below or store it in Colab Secrets under the name `FINNHUB_API_KEY`.

In [1]:
!pip -q install yfinance plotly requests pandas

## 1. Settings

In [2]:
import os, time, datetime as dt
import requests
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TICKER = "CRL"                      # ticker for price and Finnhub news
GDELT_QUERY = '"Charles River Laboratories"'         # keyword query used only for the GDELT fallback (e.g. the company name)
LOOKBACK_DAYS = 365
DATA_MODE = "auto"     # "auto" = try live APIs and fall back to the snapshot | "live" = live only | "snapshot" = no web requests
USE_CACHE = True       # reuse CSVs saved by an earlier successful live run

# Free key from https://finnhub.io/register. Leave blank to use the GDELT fallback.
FINNHUB_API_KEY = ""
try:  # Colab Secrets (key icon in the left sidebar)
    from google.colab import userdata
    FINNHUB_API_KEY = FINNHUB_API_KEY or userdata.get("FINNHUB_API_KEY") or ""
except Exception:
    FINNHUB_API_KEY = FINNHUB_API_KEY or os.environ.get("FINNHUB_API_KEY", "")

END = pd.Timestamp.today().normalize()
START = END - pd.Timedelta(days=LOOKBACK_DAYS)
used_snapshot = False
print(f"Window: {START.date()} -> {END.date()} | mode: {DATA_MODE} | "
      f"news source: {'Finnhub' if FINNHUB_API_KEY else 'GDELT'}")

Window: 2025-09-30 -> 2026-09-30 | mode: auto | news source: GDELT


### Embedded snapshot (fallback data; no need to edit)

In [3]:
import io
SNAPSHOT_CSV = """
date,close,news_count
2025-09-30,156.46,3
2025-10-01,170.48,0
2025-10-02,175.36,3
2025-10-03,177.33,2
2025-10-04,177.33,1
2025-10-05,177.33,1
2025-10-06,174.95,1
2025-10-07,170.1,4
2025-10-08,173.47,0
2025-10-09,172.14,1
2025-10-10,167.03,3
2025-10-11,167.03,2
2025-10-12,167.03,0
2025-10-13,170.64,1
2025-10-14,170.13,1
2025-10-15,170.72,0
2025-10-16,174.95,1
2025-10-17,178.52,3
2025-10-18,178.52,0
2025-10-19,178.52,0
2025-10-20,184.14,1
2025-10-21,188.26,2
2025-10-22,187.0,1
2025-10-23,190.97,0
2025-10-24,197.04,2
2025-10-25,197.04,0
2025-10-26,197.04,0
2025-10-27,195.28,1
2025-10-28,187.93,0
2025-10-29,180.15,0
2025-10-30,179.11,0
2025-10-31,180.07,0
2025-11-01,180.07,0
2025-11-02,180.07,0
2025-11-03,178.86,2
2025-11-04,177.85,0
2025-11-05,167.7,2
2025-11-06,170.88,4
2025-11-07,168.05,3
2025-11-08,168.05,2
2025-11-09,168.05,0
2025-11-10,168.97,2
2025-11-11,173.5,2
2025-11-12,171.5,0
2025-11-13,169.21,0
2025-11-14,167.81,0
2025-11-15,167.81,0
2025-11-16,167.81,1
2025-11-17,162.74,1
2025-11-18,162.53,0
2025-11-19,161.64,0
2025-11-20,162.08,3
2025-11-21,167.64,1
2025-11-22,167.64,1
2025-11-23,167.64,0
2025-11-24,170.44,0
2025-11-25,178.05,0
2025-11-26,178.94,0
2025-11-27,178.94,0
2025-11-28,178.14,0
2025-11-29,178.14,1
2025-11-30,178.14,1
2025-12-01,176.09,0
2025-12-02,174.76,1
2025-12-03,184.67,1
2025-12-04,184.04,0
2025-12-05,184.31,0
2025-12-06,184.31,0
2025-12-07,184.31,0
2025-12-08,187.12,1
2025-12-09,185.99,1
2025-12-10,194.85,3
2025-12-11,194.43,0
2025-12-12,193.03,2
2025-12-13,193.03,3
2025-12-14,193.03,3
2025-12-15,198.15,6
2025-12-16,195.22,1
2025-12-17,195.98,4
2025-12-18,193.85,4
2025-12-19,197.06,0
2025-12-20,197.06,2
2025-12-21,197.06,3
2025-12-22,203.02,1
2025-12-23,202.87,2
2025-12-24,203.16,1
2025-12-25,203.16,0
2025-12-26,203.66,0
2025-12-27,203.66,1
2025-12-28,203.66,0
2025-12-29,203.16,1
2025-12-30,201.98,0
2025-12-31,199.48,2
2026-01-01,199.48,1
2026-01-02,202.42,0
2026-01-03,202.42,0
2026-01-04,202.42,1
2026-01-05,208.32,1
2026-01-06,216.11,0
2026-01-07,216.77,1
2026-01-08,215.51,1
2026-01-09,220.91,3
2026-01-10,220.91,0
2026-01-11,220.91,0
2026-01-12,218.82,0
2026-01-13,218.99,4
2026-01-14,216.39,0
2026-01-15,223.47,1
2026-01-16,219.7,1
2026-01-17,219.7,0
2026-01-18,219.7,0
2026-01-19,219.7,0
2026-01-20,218.19,0
2026-01-21,226.77,0
2026-01-22,224.03,1
2026-01-23,219.61,0
2026-01-24,219.61,0
2026-01-25,219.61,0
2026-01-26,222.35,1
2026-01-27,219.53,1
2026-01-28,216.88,0
2026-01-29,211.68,1
2026-01-30,210.48,0
2026-01-31,210.48,0
2026-02-01,210.48,0
2026-02-02,213.22,0
2026-02-03,201.41,0
2026-02-04,201.62,1
2026-02-05,183.7,0
2026-02-06,189.21,0
2026-02-07,189.21,1
2026-02-08,189.21,2
2026-02-09,185.7,1
2026-02-10,181.24,0
2026-02-11,165.98,0
2026-02-12,158.74,0
2026-02-13,161.35,0
2026-02-14,161.35,0
2026-02-15,161.35,2
2026-02-16,161.35,2
2026-02-17,158.53,2
2026-02-18,158.0,3
2026-02-19,164.24,1
2026-02-20,168.92,1
2026-02-21,168.92,0
2026-02-22,168.92,1
2026-02-23,163.52,2
2026-02-24,166.06,1
2026-02-25,167.56,1
2026-02-26,175.46,1
2026-02-27,178.49,0
2026-02-28,178.49,0
2026-03-01,178.49,0
2026-03-02,178.61,0
2026-03-03,175.69,2
2026-03-04,179.58,0
2026-03-05,181.32,0
2026-03-06,174.46,1
2026-03-07,174.46,0
2026-03-08,174.46,0
2026-03-09,176.99,0
2026-03-10,167.37,1
2026-03-11,168.91,1
2026-03-12,152.66,5
2026-03-13,155.46,1
2026-03-14,155.46,0
2026-03-15,155.46,0
2026-03-16,151.97,3
2026-03-17,153.71,2
2026-03-18,157.11,0
2026-03-19,153.05,4
2026-03-20,153.6,2
2026-03-21,153.6,2
2026-03-22,153.6,11
2026-03-23,158.0,2
2026-03-24,159.19,1
2026-03-25,165.89,0
2026-03-26,165.13,0
2026-03-27,159.16,1
2026-03-28,159.16,0
2026-03-29,159.16,0
2026-03-30,159.72,0
2026-03-31,172.5,1
2026-04-01,174.9,1
2026-04-02,173.89,0
2026-04-03,173.89,0
2026-04-04,173.89,0
2026-04-05,173.89,0
2026-04-06,168.22,2
2026-04-07,169.54,0
2026-04-08,175.07,1
2026-04-09,175.55,2
2026-04-10,175.65,1
2026-04-11,175.65,0
2026-04-12,175.65,0
2026-04-13,178.34,0
2026-04-14,179.31,1
2026-04-15,180.56,2
2026-04-16,176.94,0
2026-04-17,183.88,0
2026-04-18,183.88,0
2026-04-19,183.88,1
2026-04-20,185.54,0
2026-04-21,183.1,1
2026-04-22,184.8,2
2026-04-23,167.74,1
2026-04-24,169.8,0
2026-04-25,169.8,1
2026-04-26,169.8,0
2026-04-27,171.23,0
2026-04-28,166.79,0
2026-04-29,163.84,0
2026-04-30,166.97,1
2026-05-01,165.78,0
2026-05-02,165.78,1
2026-05-03,165.78,1
2026-05-04,172.32,0
2026-05-05,185.39,0
2026-05-06,181.73,1
2026-05-07,181.68,3
2026-05-08,177.62,1
2026-05-09,177.62,2
2026-05-10,177.62,1
2026-05-11,168.46,1
2026-05-12,168.69,1
2026-05-13,162.85,6
2026-05-14,159.12,0
2026-05-15,150.85,0
2026-05-16,150.85,0
2026-05-17,150.85,1
2026-05-18,149.93,0
2026-05-19,152.43,0
2026-05-20,156.03,0
2026-05-21,158.86,0
2026-05-22,160.3,2
2026-05-23,160.3,0
2026-05-24,160.3,0
2026-05-25,160.3,0
2026-05-26,156.73,0
2026-05-27,164.49,1
2026-05-28,181.34,0
2026-05-29,180.71,0
2026-05-30,180.71,1
2026-05-31,180.71,0
2026-06-01,182.07,0
2026-06-02,174.79,1
2026-06-03,179.87,0
2026-06-04,185.79,1
2026-06-05,181.34,0
2026-06-06,181.34,0
2026-06-07,181.34,0
2026-06-08,186.44,0
2026-06-09,189.85,0
2026-06-10,186.29,0
2026-06-11,188.06,0
2026-06-12,187.51,0
2026-06-13,187.51,0
2026-06-14,187.51,0
2026-06-15,188.47,0
2026-06-16,184.79,0
2026-06-17,182.34,0
2026-06-18,185.0,2
2026-06-19,185.0,1
2026-06-20,185.0,0
2026-06-21,185.0,0
2026-06-22,181.56,1
2026-06-23,186.6,1
2026-06-24,202.1,7
2026-06-25,212.71,1
2026-06-26,215.75,1
2026-06-27,215.75,0
2026-06-28,215.75,0
2026-06-29,225.57,1
2026-06-30,226.79,0
2026-07-01,229.09,0
2026-07-02,230.69,0
2026-07-03,230.69,0
2026-07-04,230.69,0
2026-07-05,230.69,0
2026-07-06,231.43,0
2026-07-07,228.68,0
2026-07-08,223.96,2
2026-07-09,233.6,1
2026-07-10,233.41,1
2026-07-11,233.41,0
2026-07-12,233.41,0
2026-07-13,229.75,1
2026-07-14,231.21,1
2026-07-15,228.22,0
2026-07-16,229.16,1
2026-07-17,224.35,0
2026-07-18,224.35,0
2026-07-19,224.35,0
2026-07-20,219.7,0
2026-07-21,219.4,1
2026-07-22,218.7,0
2026-07-23,229.29,0
2026-07-24,226.58,0
2026-07-25,226.58,0
2026-07-26,226.58,0
2026-07-27,224.3,0
2026-07-28,240.43,0
2026-07-29,234.8,0
2026-07-30,234.62,2
2026-07-31,232.51,2
2026-08-01,232.51,0
2026-08-02,232.51,0
2026-08-03,230.87,0
2026-08-04,234.12,2
2026-08-05,260.72,4
2026-08-06,265.51,4
2026-08-07,267.49,1
2026-08-08,267.49,0
2026-08-09,267.49,0
2026-08-10,277.0,2
2026-08-11,282.0,0
2026-08-12,284.37,1
2026-08-13,281.27,1
2026-08-14,280.05,3
2026-08-15,280.05,0
2026-08-16,280.05,0
2026-08-17,286.64,0
2026-08-18,279.19,1
2026-08-19,290.0,0
2026-08-20,291.45,1
2026-08-21,295.19,0
2026-08-22,295.19,0
2026-08-23,295.19,0
2026-08-24,298.58,0
2026-08-25,297.64,1
2026-08-26,295.1,0
2026-08-27,296.41,1
2026-08-28,291.5,0
2026-08-29,291.5,0
2026-08-30,291.5,0
2026-08-31,288.24,1
2026-09-01,282.34,0
2026-09-02,291.76,0
2026-09-03,293.35,0
2026-09-04,288.5,0
2026-09-05,288.5,0
2026-09-06,288.5,0
2026-09-07,288.5,0
2026-09-08,280.76,0
2026-09-09,278.35,0
2026-09-10,273.0,1
2026-09-11,278.26,1
2026-09-12,278.26,0
2026-09-13,278.26,0
2026-09-14,273.11,0
2026-09-15,272.88,0
2026-09-16,273.41,0
2026-09-17,281.67,0
2026-09-18,278.06,0
2026-09-19,278.06,0
2026-09-20,278.06,0
2026-09-21,279.01,1
2026-09-22,285.11,2
2026-09-23,277.33,0
2026-09-24,294.44,3
2026-09-25,292.84,0
2026-09-26,292.84,0
2026-09-27,292.84,2
2026-09-28,294.59,0
2026-09-29,295.96,0
2026-09-30,296.54,0
"""
snapshot = pd.read_csv(io.StringIO(SNAPSHOT_CSV.strip()), parse_dates=["date"], index_col="date")
print(f"Snapshot: {snapshot.index.min().date()} -> {snapshot.index.max().date()}, {len(snapshot)} rows")

Snapshot: 2025-09-30 -> 2026-09-30, 366 rows


## 2. Daily price

In [4]:
PRICE_CACHE = f"{TICKER}_price_cache.csv"

def fetch_price():
    px = yf.download(TICKER, start=START, end=END + pd.Timedelta(days=1), auto_adjust=True, progress=False)
    if px.empty:
        raise ValueError(f"No price data for '{TICKER}' (bad ticker or Yahoo rate limit).")
    if isinstance(px.columns, pd.MultiIndex):   # newer yfinance returns (field, ticker) columns
        px.columns = px.columns.get_level_values(0)
    p = px["Close"].rename("close")
    p.index = pd.to_datetime(p.index).tz_localize(None).normalize()
    return p

if DATA_MODE != "snapshot" and USE_CACHE and os.path.exists(PRICE_CACHE):
    price = pd.read_csv(PRICE_CACHE, parse_dates=["date"], index_col="date")["close"]
    print("Price: loaded from cache")
elif DATA_MODE == "snapshot":
    price = None
else:
    try:
        price = fetch_price()
        price.rename_axis("date").to_csv(PRICE_CACHE)
        print(f"Price: {len(price)} trading days downloaded")
    except Exception as e:
        if DATA_MODE == "live":
            raise
        print(f"Price download failed ({e}); using snapshot."); price = None

if price is None:
    price = snapshot["close"]; used_snapshot = True
price.tail()

Price: loaded from cache


date
2026-09-24    294.440002
2026-09-25    292.839996
2026-09-28    294.589996
2026-09-29    295.959991
2026-09-30    295.795013
Name: close, dtype: float64

## 3. Daily news article counts

In [5]:
def finnhub_news(symbol, start, end, key, chunk_days=7):
    """Fetch Finnhub company news in weekly chunks (each response is capped), dedupe by article id."""
    rows, seen = [], set()
    cur = start
    while cur <= end:
        nxt = min(cur + pd.Timedelta(days=chunk_days - 1), end)
        r = requests.get("https://finnhub.io/api/v1/company-news",
                         params={"symbol": symbol, "from": cur.strftime("%Y-%m-%d"),
                                 "to": nxt.strftime("%Y-%m-%d"), "token": key}, timeout=30)
        if r.status_code == 429:          # rate limit (60/min on the free tier): wait and retry
            time.sleep(5); continue
        r.raise_for_status()
        for a in r.json():
            if a.get("id") in seen:
                continue
            seen.add(a.get("id"))
            rows.append({"datetime": pd.to_datetime(a["datetime"], unit="s"),
                         "headline": a.get("headline"), "source": a.get("source"), "url": a.get("url")})
        cur = nxt + pd.Timedelta(days=1)
        time.sleep(1.1)
    return pd.DataFrame(rows)


def gdelt_counts(query, start, end):
    """Daily raw article counts from GDELT DOC 2.0 (no key). Uses timelinevolraw mode."""
    r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc",
                     params={"query": query, "mode": "timelinevolraw", "format": "json",
                             "startdatetime": start.strftime("%Y%m%d000000"),
                             "enddatetime": end.strftime("%Y%m%d235959"),
                             "timelinesmooth": 0}, timeout=60)
    for wait in ((10, 20, 40, 60) if DATA_MODE == "live" else (10,)):          # GDELT allows ~1 request / 5s; back off on 429
        if r.status_code != 429:
            break
        print(f"GDELT rate limit, retrying in {wait}s..."); time.sleep(wait)
        r = requests.get(r.url, timeout=60)
    r.raise_for_status()
    data = r.json().get("timeline", [{}])[0].get("data", [])
    s = pd.Series({pd.to_datetime(d["date"]).tz_localize(None).normalize(): d["value"] for d in data}, dtype=float)
    return s.groupby(level=0).sum()


NEWS_CACHE = f"{TICKER}_news_cache_{'finnhub' if FINNHUB_API_KEY else 'gdelt'}.csv"
articles, news = pd.DataFrame(), None

if DATA_MODE != "snapshot" and USE_CACHE and os.path.exists(NEWS_CACHE):
    news = pd.read_csv(NEWS_CACHE, parse_dates=["date"], index_col="date")["news_count"]
    print("News: loaded from cache")
elif DATA_MODE != "snapshot":
    try:
        if FINNHUB_API_KEY:
            articles = finnhub_news(TICKER, START, END, FINNHUB_API_KEY)
            print(f"News: {len(articles)} unique Finnhub articles")
            news = (articles.assign(date=articles["datetime"].dt.normalize()).groupby("date").size()
                    if not articles.empty else pd.Series(dtype=float))
        else:
            news = gdelt_counts(GDELT_QUERY, START, END)
            print(f"News: {int(news.sum())} GDELT articles")
        news.rename("news_count").rename_axis("date").to_csv(NEWS_CACHE)
    except Exception as e:
        if DATA_MODE == "live":
            raise
        print(f"News download failed ({e}); using snapshot."); news = None

if news is None:
    news = snapshot["news_count"]; used_snapshot = True

news = news.rename("news_count")
news.tail()

GDELT rate limit, retrying in 10s...
News download failed (429 Client Error: Too Many Requests for url: https://api.gdeltproject.org/api/v2/doc/doc?query=%22Charles+River+Laboratories%22&mode=timelinevolraw&format=json&startdatetime=20250930000000&enddatetime=20260930235959&timelinesmooth=0); using snapshot.


date
2026-09-26    0
2026-09-27    2
2026-09-28    0
2026-09-29    0
2026-09-30    0
Name: news_count, dtype: int64

## 4. Put both series on one daily index

Both series use the same calendar-day index from `START` to `END`, so they begin and end on the same dates.
Weekends and holidays carry the last close forward, and days with no articles count as 0.
Set `TRADING_DAYS_ONLY = True` to keep only market days. News from non-trading days is then added to the next trading day.

In [6]:
TRADING_DAYS_ONLY = False

if used_snapshot:   # keep both series on the snapshot's window so they start/end together
    START, END = snapshot.index.min(), snapshot.index.max()
    print(f"Using snapshot window {START.date()} -> {END.date()}")

idx = pd.date_range(START, END, freq="D")
df = pd.DataFrame(index=idx)
df.index.name = "date"
df["close"] = price.reindex(idx).ffill().bfill()
df["news_count"] = news.reindex(idx).fillna(0).astype(int)

if TRADING_DAYS_ONLY:
    trade_days = price.index
    # map each calendar day to the next trading day on or after it
    bucket = pd.Series(trade_days, index=trade_days).reindex(idx, method="bfill")
    df = (df.assign(bucket=bucket.values).dropna(subset=["bucket"])
            .groupby("bucket").agg(close=("close", "last"), news_count=("news_count", "sum")))
    df.index.name = "date"

# relative measures
df["price_rebased_100"] = 100 * df["close"] / df["close"].iloc[0]
df["daily_return_%"] = df["close"].pct_change() * 100
df["news_7d_avg"] = df["news_count"].rolling(7, min_periods=1).mean()

print(f"{df.index.min().date()} -> {df.index.max().date()}  ({len(df)} rows)")
print(f"Total articles: {df['news_count'].sum()}  |  days with news: {(df['news_count'] > 0).sum()}")

Using snapshot window 2025-09-30 -> 2026-09-30
2025-09-30 -> 2026-09-30  (366 rows)
Total articles: 285  |  days with news: 162


## 5. Data table

In [7]:
pd.set_option("display.max_rows", 400)
display(df.round(2))

,close,news_count,price_rebased_100,daily_return_%,news_7d_avg
date,,,,,
2025-09-30,156.46,3,100.00,NaN,3.00
2025-10-01,170.48,0,108.96,8.96,1.50
2025-10-02,175.36,3,112.08,2.86,2.00
2025-10-03,177.33,2,113.34,1.12,2.00
2025-10-04,177.33,1,113.34,0.00,1.80
2025-10-05,177.33,1,113.34,0.00,1.67
2025-10-06,174.95,1,111.82,-1.34,1.57
2025-10-07,170.10,4,108.72,-2.77,1.71
2025-10-08,173.47,0,110.87,1.98,1.71


In [8]:
# Optional: save to CSV (in Colab, download it from the Files pane)
df.round(4).to_csv(f"{TICKER}_price_vs_news.csv")

PermissionError: [Errno 13] Permission denied: 'CRL_price_vs_news.csv'

## 6. Chart: price (left axis) vs. news volume (right axis)

In [9]:
USE_REBASED = True   # True: price rebased to 100 at the start; False: raw closing price

y_price = df["price_rebased_100"] if USE_REBASED else df["close"]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df.index, y=y_price, name="Price (rebased=100)" if USE_REBASED else "Close",
                         line=dict(color="#1f77b4", width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=df.index, y=df["news_count"], name="Daily articles",
                         line=dict(color="#ff7f0e", width=1), opacity=0.45), secondary_y=True)
fig.add_trace(go.Scatter(x=df.index, y=df["news_7d_avg"], name="Articles (7-day avg)",
                         line=dict(color="#d62728", width=2)), secondary_y=True)
fig.update_layout(title=f"{TICKER}: daily price vs. news volume ({df.index.min().date()} to {df.index.max().date()})",
                  hovermode="x unified", height=550, legend=dict(orientation="h", y=-0.15))
fig.update_xaxes(range=[df.index.min(), df.index.max()])
fig.update_yaxes(title_text="Price (rebased=100)" if USE_REBASED else "Close ($)", secondary_y=False)
fig.update_yaxes(title_text="News articles / day", secondary_y=True, rangemode="tozero")
fig.show()

## 7. (Optional) Quick relationship check

In [10]:
tmp = df.dropna(subset=["daily_return_%"])
print("Corr(news_count, daily return)      :", round(tmp["news_count"].corr(tmp["daily_return_%"]), 3))
print("Corr(news_count, |daily return|)    :", round(tmp["news_count"].corr(tmp["daily_return_%"].abs()), 3))
print("Corr(news 7d avg, rebased price)    :", round(df["news_7d_avg"].corr(df["price_rebased_100"]), 3))
if not articles.empty:
    display(articles.sort_values("datetime", ascending=False).head(20))

Corr(news_count, daily return)      : 0.05
Corr(news_count, |daily return|)    : 0.191
Corr(news 7d avg, rebased price)    : -0.251
